# Create Russian Foundation for Basic Research Awards (GRANT PATTERN, legacy portal archive)

The Russian Foundation for Basic Research (Российский фонд фундаментальных
исследований, RFBR/РФФИ) was Russia's basic-science grant funder from 1992
until its 2022 merger into the Russian Science Foundation. Its legacy grant
archive survives on the public portal `www.rfbr.ru/project_search`
(maintained by РЦНИ/RCSI). Probed 2026-07-12:

- `kias.rfbr.ru` — reachable but login-gated (КИАС РЦНИ), no public catalog.
- `search.rfbr.ru` — DEAD; rfbr.ru still links to it as the "2012+" search.
- `www.rfbr.ru/project_search` — WORKS. Dense coverage for grants with
  "Год проведения" **1993-2018**; 2019 has only 14 rows; **2020+ is empty**.
  RFBR grants from 2019-2021 are NOT publicly harvestable anywhere found
  (documented gap; OpenAlex keeps its Crossref-derived stubs for those).

The source script `scripts/local/rfbr_to_s3.py` runs a two-phase harvest:
1. enumerate ALL listing pages per year (~21k pages x 20 rows) -> native
   grant number, Russian title, year, research area, contest type, and
   application status;
2. fetch detail pages to enrich with PI (Руководитель) + abstract for the
   grants OpenAlex already cites (~13k in covered years). Detail-enriching
   all ~250k funded rows is deferred (script flag `--details-all`).

**The archive MIXES funded grants and rejected applications** (~61% funded
in a random page sample). The script keeps ONLY rows with
`Статус заявки = 'поддержана'` (supported/funded) — rejected applications
are not awards.

**Awarding body:** Russian Foundation for Basic Research — F4320321079 (RU).
Path A (F4320* Crossref-registered funder, in `openalex.common.funder`).

**Schema choices / known limitations:**
- `funder_award_id` = the native RFBR grant number, kept **verbatim**
  including Cyrillic contest suffixes where the portal shows them.
- **NO amounts anywhere** on the portal -> `amount`/`currency` NULL on every
  row. **Step 6.7 amount check WAIVED** (source publishes no amounts).
- **No host organization** is published -> `lead_investigator.affiliation`
  carries only `country = 'RU'` with a NULL name.
- **Dates**: only "Год проведения" (execution year) is published ->
  `start_year` only; `start_date`/`end_date`/`end_year` NULL.
- **PI**: present only on detail-enriched rows (~OpenAlex-cited subset, plus
  older records that publish initials-only names, e.g. `Прохоров В. В.`).
  Russian order Family Given Patronymic; family = first token. No ORCIDs.
- `description` (abstract) present only on detail-enriched rows; pre-~2000
  detail pages publish no abstract at all.
- `funder_scheme` = contest type string (e.g. «(а) инициативные проекты»);
  `funding_type = 'research'`.
- **Source-data title artifacts (kept verbatim):** ~2.4k rows from 1996
  \"гранты поддержки ведущих научных школ\" carry placeholder titles
  (`вннн` x1,688, `вгнн` x753) **in the portal's own database** (verified
  against live detail pages, e.g. `/project_search/79003/`). Another ~700
  rows share the genuine repeated program title «Доступ к электронным
  научным информационным ресурсам зарубежных издательств» (national
  e-subscription program). These are real funded grants with valid grant
  numbers; they are NOT scraper bugs and are kept as published.

**Prerequisites:** run `scripts/local/rfbr_to_s3.py` first (checkpointed,
resumable; ~21k listing pages + ~13k detail fetches).

**Data source:** https://www.rfbr.ru/project_search
**S3 location:** `s3a://openalex-ingest/awards/rfbr/rfbr_projects.parquet`


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rfbr_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rfbr/rfbr_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.rfbr_raw;

## Step 1.5: Inspect raw + money/currency scan

Per runbook §1.5, scan every column for money-shaped data even though the
source is known to publish none. Expected result: **zero money-flavored
columns** — the RFBR portal carries no amounts (see header; §6.7 waived).

In [ ]:
%sql
DESCRIBE openalex.awards.rfbr_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.rfbr_raw LIMIT 5;

In [ ]:
%sql
-- Money/currency-flavored column scan (runbook §1.5). Expect 0 rows.
SELECT column_name FROM (DESCRIBE openalex.awards.rfbr_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp|currenc|ccy|iso_4217';

In [ ]:
%sql
-- Status sanity: build keeps only supported rows; confirm nothing else leaked.
SELECT status, COUNT(*) AS n FROM openalex.awards.rfbr_raw GROUP BY status;

## Step 1.6: Fail-fast — verify RFBR funder row exists (Path A, F4320*)

RFBR is F4320321079 (Crossref-registered), so `openalex.common.funder` MUST
return exactly 1 row — if it doesn't, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321079;  -- Russian Foundation for Basic Research (expect exactly 1 row)

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rfbr_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321079  -- Russian Foundation for Basic Research
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(r.funder_award_id)
    ))) % 9000000000 AS id,
    r.display_name,                                 -- Russian project title (verbatim)
    r.description,                                  -- Russian abstract (detail-enriched rows only)
    f.funder_id,
    r.funder_award_id,                              -- native RFBR grant number (verbatim)
    CAST(NULL AS DOUBLE) AS amount,                 -- portal publishes no amounts (§6.7 waived)
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    r.funder_scheme,                                -- contest type (Russian)
    'rfbr' AS provenance,
    CAST(NULL AS DATE) AS start_date,               -- only the year is published
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(r.start_year AS INT) AS start_year,    -- "Год проведения"
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN r.lead_family_name IS NULL OR r.lead_family_name = '' THEN NULL
        ELSE struct(
            NULLIF(TRIM(r.lead_given_name), '') AS given_name,
            TRIM(r.lead_family_name) AS family_name,
            CAST(NULL AS STRING) AS orcid,          -- not published
            CAST(NULL AS DATE) AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,       -- portal publishes no host org
                'RU' AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING,
        ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING,
        ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) AS investigators,
    r.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(r.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.rfbr_raw r
CROSS JOIN funder_resolved f
WHERE r.funder_award_id IS NOT NULL
  AND r.display_name IS NOT NULL
  AND r.status = 'поддержана';  -- belt-and-braces: funded rows only

## Step 3: Insert into openalex_awards_raw at priority 401

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rfbr' AND priority = 401;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    401 as priority  -- RFBR priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.rfbr_awards;

## Step 6: Verification

Full §6.1–6.8. **§6.7 amount check is WAIVED**: the RFBR portal publishes no
per-project amounts — expect 0% amount coverage by design. PI coverage is
partial by design (detail-enriched subset only; see header).

In [ ]:
%sql
SELECT COUNT(*) AS total_rfbr_award_rows FROM openalex.awards.rfbr_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.rfbr_awards;

In [ ]:
%sql
-- §6.3 Data completeness (PI/description partial by design — see header)
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_year) AS has_start_year,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_year) * 100.0 / COUNT(*), 1) AS pct_start_year,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) AS pct_pi
FROM openalex.awards.rfbr_awards;

In [ ]:
%sql
-- §6.7 amount check — WAIVED: RFBR portal publishes no per-project amounts.
-- Expect has_amount = 0 and distinct_currencies = 0 BY DESIGN.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.rfbr_awards;

In [ ]:
%sql
-- §6.4 sample inspection
SELECT id, SUBSTRING(display_name, 1, 60) AS title, funder_award_id,
       lead_investigator.given_name, lead_investigator.family_name,
       start_year, SUBSTRING(funder_scheme, 1, 50) AS scheme
FROM openalex.awards.rfbr_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a PI frequency check — long-tail expected on the enriched subset;
-- prolific RFBR PIs legitimately hold multiple grants (counts up to ~10).
SELECT lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       COUNT(*) AS n
FROM openalex.awards.rfbr_awards
WHERE lead_investigator IS NOT NULL
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a display_name frequency. KNOWN SOURCE ARTIFACTS (see header): the
-- 1996 leading-science-schools program carries placeholder titles in the
-- portal's own DB ('вннн' ~1,688, 'вгнн' ~753; verified on live detail
-- pages), and ~700 rows share the genuine national e-subscription program
-- title. Anything ELSE with hundreds of repeats would be a real bug.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.rfbr_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- §6.6 year distribution — expect 1993-2018 dense, 2019 tiny, nothing after
SELECT start_year, COUNT(*) AS cnt
FROM openalex.awards.rfbr_awards
GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
-- §6.5 funder consistency — should be exactly RFBR
SELECT funder.id, funder.display_name, funder_id, COUNT(*) AS n
FROM openalex.awards.rfbr_awards
GROUP BY 1, 2, 3;

In [ ]:
%sql
-- §6.8 confirm rows reached the shared raw table at the right priority
SELECT provenance, priority, COUNT(*) AS n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rfbr'
GROUP BY provenance, priority;